# Module 8 • Large Language Models

# Lesson 48 • LLM Agents, Tool Use, and Function Calling

**Course:** Natural Language Processing: From Foundations to Large Language Models  
**Author:** Eman Khater  
**Difficulty:** Advanced  
**Execution target:** CPU only

---

## Scope

This lesson introduces tool-augmented LLM systems and agentic workflows. The executable core is fully offline and demonstrates tool schemas, structured arguments, validation, tool registries, action-observation loops, multi-step planning, error recovery, grounding, traces, and agent evaluation. The local planner is deterministic so the notebook requires no API key or external model download.

## Learning Objectives

After completing this lesson, the learner should be able to:

- distinguish ordinary generation from tool-augmented generation;
- define and validate tool schemas;
- build a tool registry;
- implement an action-observation loop;
- handle single-step and multi-step tasks;
- recover from invalid arguments and tool failures;
- ground final answers in tool observations;
- log agent traces;
- evaluate tool selection, argument construction, task success, and step efficiency;
- discuss security, side effects, least privilege, multilingual tool use, and Arabic function calling.

## Table of Contents

1. What Is an LLM Agent?
2. Tool-Augmented Language Models
3. Function Calling
4. Tool Schemas
5. Argument Validation
6. Tool Registries
7. Action–Observation Loops
8. Single-Step and Multi-Step Tasks
9. Planning
10. Offline Planner
11. Calculator Tool
12. Search Tool
13. Lookup Tool
14. Tool Execution
15. Agent State
16. Agent Loop
17. Maximum-Step Limits
18. Error Recovery
19. Unknown Tools
20. Invalid Arguments
21. Tool Failures
22. Retry Policies
23. Final Answer Grounding
24. Trace Logging
25. Structured Traces
26. Offline Agent Tasks
27. Tool-Selection Evaluation
28. Argument Accuracy
29. Task Success
30. Step Efficiency
31. Failure Taxonomy
32. Tool Hallucination
33. Prompt Injection and Tool Security
34. Least Privilege
35. Side Effects and Confirmation
36. Parallel Versus Sequential Calls
37. Caching
38. Idempotency
39. Agent Memory
40. Retrieval as a Tool
41. Code Execution as a Tool
42. Human-in-the-Loop
43. Multilingual Tool Use
44. Arabic Function Calling
45. Reproducibility
46. Knowledge Check
47. Exercises
48. Summary and Next Lesson

# 1. What Is an LLM Agent?

An agentic LLM system combines a language model or planner with an execution loop that can choose actions, call tools, observe results, and continue until a task is complete.

In [ ]:
import json
import operator
import platform
import re
from dataclasses import dataclass, field
from typing import Any, Callable

import numpy as np
import pandas as pd

agent_components = pd.DataFrame([
    ("Planner", "chooses the next action"),
    ("Tool registry", "defines available actions"),
    ("Executor", "runs selected tools"),
    ("Observation handler", "returns results"),
    ("State", "stores progress"),
    ("Stop condition", "prevents endless loops"),
], columns=["Component", "Role"])
agent_components

# 2. Tool-Augmented Language Models

Tools extend a language model with capabilities better handled externally, such as arithmetic, retrieval, databases, calendars, code execution, and structured APIs.

# 3. Function Calling

Function calling is a structured interface in which the model selects a named tool and supplies arguments that conform to a schema.

# 4. Tool Schemas

A tool schema should define its name, purpose, arguments, types, and required fields.

In [ ]:
calculator_schema = {
    "name": "calculator",
    "description": "Evaluate a basic arithmetic expression.",
    "parameters": {
        "type": "object",
        "properties": {"expression": {"type": "string"}},
        "required": ["expression"],
    },
}
calculator_schema

# 5. Argument Validation

Arguments should be validated before a tool runs.

In [ ]:
def validate_required_fields(arguments: dict, required_fields: list[str]):
    missing = [name for name in required_fields if name not in arguments]
    return len(missing) == 0, missing

validate_required_fields({"expression": "2 + 2"}, ["expression"])

# 6. Tool Registries

A tool registry maps canonical tool names to executable functions and metadata.

In [ ]:
@dataclass
class Tool:
    name: str
    description: str
    function: Callable[..., Any]
    required_fields: list[str]

tool_registry: dict[str, Tool] = {}

# 7. Action–Observation Loops

A typical loop is: state → choose action → execute tool → observe result → update state → repeat.

# 8. Single-Step and Multi-Step Tasks

A calculation can be single-step. A request such as “find Alpha's value and multiply it by 3” requires at least two dependent actions.

# 9. Planning

Planning decomposes a task into actions. The engineering goal is not verbose reasoning; it is correct, safe, and efficient action selection.

# 10. Offline Planner

The local planner uses deterministic rules for reproducibility.

In [ ]:
def normalize_query(text: str) -> str:
    return re.sub(r"\s+", " ", text.lower().strip())

def detect_arithmetic_expression(text: str):
    match = re.search(r"(-?\d+(?:\.\d+)?)\s*([+\-*/])\s*(-?\d+(?:\.\d+)?)", text)
    return " ".join(match.groups()) if match else None

detect_arithmetic_expression("Please calculate 14 * 9.")

# 11. Calculator Tool

In [ ]:
SAFE_OPERATORS = {"+": operator.add, "-": operator.sub, "*": operator.mul, "/": operator.truediv}

def calculator(expression: str) -> dict:
    match = re.fullmatch(r"\s*(-?\d+(?:\.\d+)?)\s*([+\-*/])\s*(-?\d+(?:\.\d+)?)\s*", expression)
    if not match:
        return {"ok": False, "error": "Only one basic binary arithmetic expression is supported."}
    left_text, symbol, right_text = match.groups()
    left, right = float(left_text), float(right_text)
    if symbol == "/" and right == 0:
        return {"ok": False, "error": "division by zero"}
    return {"ok": True, "result": SAFE_OPERATORS[symbol](left, right)}

calculator("14 * 9")

# 12. Search Tool

The offline search tool scans a tiny local document collection.

In [ ]:
search_documents = [
    {"id": "doc_1", "text": "Project Alpha is scheduled for September and has a budget value of 12."},
    {"id": "doc_2", "text": "Project Beta focuses on retrieval evaluation and has a budget value of 20."},
    {"id": "doc_3", "text": "The support handbook states that refunds are processed within five days."},
]

def search_documents_tool(query: str) -> dict:
    query_tokens = set(re.findall(r"\b\w+\b", query.lower()))
    scored = []
    for document in search_documents:
        document_tokens = set(re.findall(r"\b\w+\b", document["text"].lower()))
        scored.append((len(query_tokens & document_tokens), document))
    scored.sort(key=lambda item: item[0], reverse=True)
    score, document = scored[0]
    if score == 0:
        return {"ok": False, "error": "no matching document"}
    return {"ok": True, "document_id": document["id"], "text": document["text"]}

search_documents_tool("refund handbook")

# 13. Lookup Tool

Structured lookup can be preferable to free-text search when stable keys are known.

In [ ]:
knowledge_base = {"alpha_value": 12, "beta_value": 20, "refund_days": 5}

def lookup(key: str) -> dict:
    if key not in knowledge_base:
        return {"ok": False, "error": f"unknown key: {key}"}
    return {"ok": True, "key": key, "value": knowledge_base[key]}

lookup("alpha_value")

# 14. Tool Execution

In [ ]:
tool_registry = {
    "calculator": Tool("calculator", "Evaluate arithmetic.", calculator, ["expression"]),
    "search": Tool("search", "Search local documents.", search_documents_tool, ["query"]),
    "lookup": Tool("lookup", "Read a structured value.", lookup, ["key"]),
}

def execute_tool(tool_name: str, arguments: dict) -> dict:
    if tool_name not in tool_registry:
        return {"ok": False, "error": f"unknown tool: {tool_name}"}
    tool = tool_registry[tool_name]
    valid, missing = validate_required_fields(arguments, tool.required_fields)
    if not valid:
        return {"ok": False, "error": "missing required arguments", "missing": missing}
    try:
        return tool.function(**arguments)
    except Exception as error:
        return {"ok": False, "error": str(error)}

execute_tool("calculator", {"expression": "7 + 8"})

# 15. Agent State

In [ ]:
@dataclass
class AgentState:
    user_query: str
    step: int = 0
    observations: list[dict] = field(default_factory=list)
    trace: list[dict] = field(default_factory=list)
    finished: bool = False
    final_answer: str | None = None

# 16. Agent Loop

The planner supports arithmetic, lookup, search, and lookup-then-calculation.

In [ ]:
def plan_next_action(state: AgentState) -> dict:
    query = normalize_query(state.user_query)
    if state.observations:
        latest = state.observations[-1]
        result = latest["result"]
        if not result.get("ok"):
            return {"type": "finish", "answer": "The requested tool operation could not be completed."}
        if latest["tool"] == "lookup" and "multiply" in query:
            match = re.search(r"multiply(?: it)? by (\d+(?:\.\d+)?)", query)
            if match:
                return {"type": "tool", "tool": "calculator", "arguments": {"expression": f"{result['value']} * {match.group(1)}"}}
        if latest["tool"] == "calculator":
            return {"type": "finish", "answer": str(result["result"])}
        if latest["tool"] == "lookup":
            return {"type": "finish", "answer": str(result["value"])}
        if latest["tool"] == "search":
            return {"type": "finish", "answer": result["text"]}
    arithmetic = detect_arithmetic_expression(query)
    if arithmetic:
        return {"type": "tool", "tool": "calculator", "arguments": {"expression": arithmetic}}
    if "alpha" in query:
        return {"type": "tool", "tool": "lookup", "arguments": {"key": "alpha_value"}}
    if "beta" in query:
        return {"type": "tool", "tool": "lookup", "arguments": {"key": "beta_value"}}
    if "refund" in query:
        return {"type": "tool", "tool": "search", "arguments": {"query": query}}
    return {"type": "finish", "answer": "No available tool is appropriate for this request."}

def run_agent(user_query: str, maximum_steps: int = 5) -> AgentState:
    state = AgentState(user_query=user_query)
    while not state.finished and state.step < maximum_steps:
        state.step += 1
        action = plan_next_action(state)
        state.trace.append({"step": state.step, "action": action})
        if action["type"] == "finish":
            state.finished = True
            state.final_answer = action["answer"]
            break
        result = execute_tool(action["tool"], action["arguments"])
        observation = {"tool": action["tool"], "arguments": action["arguments"], "result": result}
        state.observations.append(observation)
        state.trace.append({"step": state.step, "observation": observation})
    if not state.finished:
        state.finished = True
        state.final_answer = "The agent stopped after reaching the maximum step limit."
    return state

run_agent("What is 14 * 9?").final_answer

# 17. Maximum-Step Limits

Every agent loop needs a bounded stopping condition to prevent runaway execution.

# 18. Error Recovery

Robust agents distinguish planning errors, schema errors, tool errors, observation errors, and stopping errors.

# 19. Unknown Tools

In [ ]:
execute_tool("weather", {"location": "Cairo"})

# 20. Invalid Arguments

In [ ]:
execute_tool("calculator", {})

# 21. Tool Failures

In [ ]:
execute_tool("calculator", {"expression": "4 / 0"})

# 22. Retry Policies

Retries should be bounded and limited to recoverable failures.

In [ ]:
def execute_with_retry(tool_name: str, arguments: dict, retries: int = 1) -> dict:
    attempts = []
    for attempt in range(retries + 1):
        result = execute_tool(tool_name, arguments)
        attempts.append({"attempt": attempt + 1, "result": result})
        if result.get("ok"):
            return {"ok": True, "attempts": attempts, "final_result": result}
    return {"ok": False, "attempts": attempts, "final_result": attempts[-1]["result"]}

execute_with_retry("calculator", {"expression": "2 + 3"}, retries=1)

# 23. Final Answer Grounding

The final answer should be derived from observations, not invented values.

# 24. Trace Logging

In [ ]:
traced = run_agent("Find the alpha value and multiply it by 3.")
pd.DataFrame([{
    "entry": i + 1,
    "content": json.dumps(event, ensure_ascii=False),
} for i, event in enumerate(traced.trace)])

# 25. Structured Traces

Structured traces help diagnose wrong tool selection, incorrect arguments, repeated calls, ignored observations, and premature stopping.

# 26. Offline Agent Tasks

In [ ]:
agent_tasks = pd.DataFrame([
    ("What is 8 + 5?", "calculator", "13.0"),
    ("What is the alpha value?", "lookup", "12"),
    ("Find the alpha value and multiply it by 2.", "lookup->calculator", "24.0"),
    ("What does the refund handbook say?", "search", "The support handbook states that refunds are processed within five days."),
], columns=["query", "expected_tool_path", "expected_answer"])
agent_tasks

# 27. Tool-Selection Evaluation

In [ ]:
def observed_tool_path(state: AgentState) -> str:
    return "->".join(obs["tool"] for obs in state.observations)

evaluation_rows = []
for row in agent_tasks.itertuples(index=False):
    state = run_agent(row.query)
    evaluation_rows.append({
        "query": row.query,
        "expected_tool_path": row.expected_tool_path,
        "observed_tool_path": observed_tool_path(state),
        "expected_answer": row.expected_answer,
        "observed_answer": state.final_answer,
    })
agent_evaluation = pd.DataFrame(evaluation_rows)
agent_evaluation["tool_path_correct"] = agent_evaluation["expected_tool_path"] == agent_evaluation["observed_tool_path"]
agent_evaluation

# 28. Argument Accuracy

In [ ]:
expected_argument_examples = [
    ("What is 8 + 5?", {"expression": "8 + 5"}),
    ("What is the alpha value?", {"key": "alpha_value"}),
]
rows=[]
for query, expected in expected_argument_examples:
    state=run_agent(query)
    observed=state.observations[0]["arguments"] if state.observations else {}
    rows.append({"query":query,"expected":expected,"observed":observed,"correct":expected==observed})
pd.DataFrame(rows)

# 29. Task Success

In [ ]:
agent_evaluation["task_success"] = agent_evaluation["expected_answer"] == agent_evaluation["observed_answer"]
pd.Series({
    "tool_path_accuracy": agent_evaluation["tool_path_correct"].mean(),
    "task_success_rate": agent_evaluation["task_success"].mean(),
})

# 30. Step Efficiency

In [ ]:
pd.DataFrame([{
    "query": row.query,
    "steps": (state := run_agent(row.query)).step,
    "tool_calls": len(state.observations),
} for row in agent_tasks.itertuples(index=False)])

# 31. Failure Taxonomy

In [ ]:
failure_taxonomy = pd.DataFrame([
    ("Tool selection", "wrong tool chosen"),
    ("Argument construction", "tool correct but arguments wrong"),
    ("Tool hallucination", "nonexistent tool requested"),
    ("Execution failure", "tool returns error"),
    ("Observation misuse", "result ignored or misread"),
    ("Looping", "agent repeats actions"),
    ("Premature finish", "agent stops too early"),
    ("Unsupported answer", "answer not grounded in observations"),
], columns=["Failure", "Description"])
failure_taxonomy

# 32. Tool Hallucination

Tool hallucination occurs when an agent invents a nonexistent tool, unsupported arguments, or nonexistent tool output.

# 33. Prompt Injection and Tool Security

Retrieved documents, webpages, emails, and tool outputs are untrusted data. Instruction-like text inside them should not automatically control the agent.

# 34. Least Privilege

Give an agent only the tools and permissions required for the current task.

# 35. Side Effects and Confirmation

Read-only tools and write tools have different risk profiles. Irreversible or consequential actions should use explicit safeguards and confirmation policies.

In [ ]:
side_effect_levels = pd.DataFrame([
    ("Read-only lookup", "low"),
    ("Search", "low"),
    ("Create draft", "medium"),
    ("Send message", "high"),
    ("Delete data", "high"),
    ("Financial transaction", "very high"),
], columns=["Tool action", "Relative side-effect risk"])
side_effect_levels

# 36. Parallel Versus Sequential Calls

Independent tool calls can run in parallel. Dependent calls must remain sequential.

# 37. Caching

Caching can reduce repeated latency and cost for stable deterministic calls. Cache invalidation is required for changing data.

# 38. Idempotency

An idempotent operation can be repeated without creating additional side effects. This property is especially useful when retries are possible.

# 39. Agent Memory

Agent memory can store conversation state, prior observations, user preferences, and task progress. Memory should be scoped to avoid stale context.

# 40. Retrieval as a Tool

Retrieval can be exposed as a first-class tool, allowing the planner to decide whether external evidence is needed.

# 41. Code Execution as a Tool

Code execution enables calculations, data analysis, transformations, and plots, but requires strong sandboxing and resource controls.

# 42. Human-in-the-Loop

Human approval can be inserted before irreversible, expensive, risky, or uncertain actions.

# 43. Multilingual Tool Use

Multilingual systems should separate user-language understanding, canonical tool selection, schema serialization, and localized final responses.

# 44. Arabic Function Calling

Arabic tool-use systems should consider Arabic-Indic digits, right-to-left display, transliteration, morphology, MSA/dialect variation, and vocalization policy.

In [ ]:
arabic_agent_examples = pd.DataFrame([
    ("اِحْسِبْ ٨ + ٥", "calculator", "normalize Arabic-Indic digits if required"),
    ("اِبْحَثْ عَنْ قِيمَةِ الْمَشْرُوعِ أَلْفَا", "lookup/search", "morphology and transliteration"),
    ("أَعِدِ النَّاتِجَ بِصِيغَةِ JSON", "structured output", "preserve requested format"),
], columns=["Arabic instruction", "Likely behavior", "Consideration"])
arabic_agent_examples

For fully vocalized Arabic workflows, tashkeel should be preserved in user-facing text when required, while internal tool arguments may use canonical identifiers.

# 45. Reproducibility

Record the tool registry, schemas, planner policy, maximum steps, retry policy, confirmation policy, tool-selection metrics, argument accuracy, task success, and trace format.

In [ ]:
reproducibility_record = pd.Series({
    "module": "Module 8 • Large Language Models",
    "lesson": "Lesson 48 • LLM Agents, Tool Use, and Function Calling",
    "tool_count": len(tool_registry),
    "evaluation_tasks": len(agent_tasks),
    "maximum_steps": 5,
    "offline_execution": True,
    "python": platform.python_version(),
}, name="Lesson 48 experiment")
reproducibility_record

# 46. Knowledge Check

1. What makes an LLM system agentic?
2. What is function calling?
3. Why are tool schemas important?
4. Why validate tool arguments?
5. What is an action-observation loop?
6. Why impose a maximum-step limit?
7. What is tool hallucination?
8. How do observations ground final answers?
9. Why separate tool-selection accuracy from argument accuracy?
10. What is task-success rate?
11. Why use least-privilege tool access?
12. Why are side-effecting tools riskier?
13. When are parallel calls appropriate?
14. What is idempotency?
15. Why can multilingual agents use canonical internal identifiers?

# 47. Exercises

## Exercise 1 — New Tool
Add a unit-conversion tool.

## Exercise 2 — Tool Schema
Add optional and required arguments.

## Exercise 3 — Multi-Step Plan
Create a task requiring search followed by calculation.

## Exercise 4 — Retry Logic
Retry only selected recoverable failures.

## Exercise 5 — Tool Hallucination
Detect an invented tool request.

## Exercise 6 — Trace Analysis
Summarize all actions and observations.

## Exercise 7 — Human Approval
Add an approval gate before a simulated write action.

## Exercise 8 — Caching
Cache deterministic lookup results.

## Exercise 9 — Arabic Agent
Add Arabic-Indic digit normalization.

## Exercise 10 — Agent Evaluation
Build a 20-task benchmark with path, argument, and answer labels.

## Challenge Exercises

1. Add parallel independent tool calls.
2. Implement replanning after tool failure.
3. Add retrieval as a first-class tool.
4. Build a tool-use confusion matrix.
5. Create a bilingual English–Arabic agent benchmark.

# 48. Summary and Next Lesson

In this lesson, agents were decomposed into planner, tools, executor, observations, state, and stopping rules. Tool schemas, validation, calculator/search/lookup tools, multi-step planning, retries, trace logging, grounding, evaluation, least privilege, side effects, caching, memory, human approval, multilingual tool use, Arabic, and tashkeel considerations were covered.

## Next Lesson

**Lesson 49: Multimodal Large Language Models and Vision-Language Foundations** introduces multimodal inputs, image-text alignment, vision encoders, projection layers, visual question answering, captioning, multimodal prompting, and multimodal evaluation.

# References

- Schick, T. et al. *Toolformer: Language Models Can Teach Themselves to Use Tools*.
- Yao, S. et al. *ReAct: Synergizing Reasoning and Acting in Language Models*.
- Qin, Y. et al. research on tool-learning benchmarks for large language models.
- Jurafsky, D., & Martin, J. H. *Speech and Language Processing*.